# Week 4, Lab 3 — LangGraph state machine

Draft → critique → revise or stop.


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 4'
LAB = 'Lab 3 — state machine'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 4 / Lab 3 — state machine
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate langchain langchain-huggingface langgraph
else:
    %pip install -q langchain langchain-ollama langgraph ollama


In [6]:
pip install grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

llm = get_langchain_llm()

class State(TypedDict):
    topic: str
    draft: str
    critique: str
    loops: int

def draft_node(state: State) -> State:
    prompt = f"Write 3 sentences for students on: {state['topic']}"
    if state.get("critique"):
        prompt += f"\nRevise using this critique: {state['critique']}"
    text = llm.invoke(prompt).content
    return {**state, "draft": text, "loops": state.get("loops", 0) + 1}

def critique_node(state: State) -> State:
    text = llm.invoke(
        f"Critique this student explainer in one sentence. If it is clear enough, reply with exactly APPROVED.\n\n{state['draft']}"
    ).content
    return {**state, "critique": text}

def should_continue(state: State) -> str:
    if "APPROVED" in (state.get("critique") or "").upper():
        return "stop"
    if state.get("loops", 0) >= 3:
        return "stop"
    return "revise"

g = StateGraph(State)
g.add_node("draft", draft_node)
g.add_node("critique", critique_node)
g.add_edge(START, "draft")
g.add_edge("draft", "critique")
g.add_conditional_edges("critique", should_continue, {"revise": "draft", "stop": END})
app = g.compile()
out = app.invoke({"topic": "ReAct agents", "draft": "", "critique": "", "loops": 0})
print("LOOPS", out["loops"])
print("CRITIQUE", out["critique"])
print("DRAFT\n", out["draft"])
print(app.get_graph().print_ascii())


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_len

LOOPS 1
CRITIQUE <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Critique this student explainer in one sentence. If it is clear enough, reply with exactly APPROVED.

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write 3 sentences for students on: ReAct agents<|im_end|>
<|im_start|>assistant
Sure! Here are three sentences that could be useful for students to react to:

1. "I can't believe I've just learned about reactivation agents! This is so exciting."

2. "Reactive agents sound pretty cool and might change the way we think about fighting crime."

3. "The idea of reactivating criminals seems like a risky proposition but also a potentially powerful tool in combating organized crime."<|im_end|>
<|im_start|>assistant
Here are three more sentences you can use:

4. "It's great to see how innovative researchers are in developing new methods for dealing wit

Always cap loops. Then run `lab4_langgraph_multiagent.ipynb` (already in this folder).
